In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import bitsandbytes
from PIL import Image
from IPython.display import display
from pathlib import Path
import unicodedata
import sys
import torch
import open_clip
import chromadb
import re
import os
import numpy as np
import subprocess
from PIL import Image
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import soundfile as sf
from datetime import timedelta

from sqlalchemy.orm import Session
from sqlalchemy import text

import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
import pillow_avif
import shutil

c:\Users\georg\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [3]:
!nvidia-smi

Wed Mar 18 17:31:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P0             25W /   85W |     101MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [5]:
NAME = "Seth (God)"
is_landmark = False
if is_landmark:
    with Session(engine) as session:
        result = session.execute(text("SELECT landmark_script FROM landmarks_scripts as ls, landmarks as l WHERE ls.landmark_id = l.id AND l.name = :name"), {"name": NAME})
        script = result.fetchall()[0][0]

else:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": NAME})
        script = result.fetchall()[0][0]

print(script)

In ancient Egypt, the mysterious god Set ruled with power and chaos. Worshipped since before recorded history, he was associated with storms, eclipses, thunderstorms, earthquakes, and the desert—making him both strong and unpredictable. His violent birth story, where he ripped himself from his mother's womb instead of being born normally, symbolized his chaotic nature.

Set held the was-sceptre and wore the Set animal as symbols, yet despite marriage to Nephthys, remained childless—a strange figure in the Egyptian pantheon. However, during some periods, Seth became one of Egypt’s great gods. His name appeared on Pharaohs like Seti I and II, who promoted him as a protector of Re, slaying Apopis with fierce power.

Yet not everyone saw his strength positively. During the Second Intermediate Period, he was associated with evil forces like the Hyksos invaders, who linked him to their storm god Baal. This perception persisted through Greek and Roman times, casting Set as an evil deity.

Set

In [6]:
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

["In ancient Egypt, the mysterious god Set ruled with power and chaos. Worshipped since before recorded history, he was associated with storms, eclipses, thunderstorms, earthquakes, and the desert—making him both strong and unpredictable. His violent birth story, where he ripped himself from his mother's womb instead of being born normally, symbolized his chaotic nature.",
 'Set held the was-sceptre and wore the Set animal as symbols, yet despite marriage to Nephthys, remained childless—a strange figure in the Egyptian pantheon. However, during some periods, Seth became one of Egypt’s great gods. His name appeared on Pharaohs like Seti I and II, who promoted him as a protector of Re, slaying Apopis with fierce power.',
 'Yet not everyone saw his strength positively. During the Second Intermediate Period, he was associated with evil forces like the Hyksos invaders, who linked him to their storm god Baal. This perception persisted through Greek and Roman times, casting Set as an evil dei

In [10]:
sentences = []
sentences_per_paragraph = []
for p in paragraphs:
    sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
    sentences_per_paragraph.append(len(sentences[-1]))
sentences

[['In ancient Egypt, the mysterious god Set ruled with power and chaos.',
  'Worshipped since before recorded history, he was associated with storms, eclipses, thunderstorms, earthquakes, and the desert—making him both strong and unpredictable.',
  "His violent birth story, where he ripped himself from his mother's womb instead of being born normally, symbolized his chaotic nature."],
 ['Set held the was-sceptre and wore the Set animal as symbols, yet despite marriage to Nephthys, remained childless—a strange figure in the Egyptian pantheon.',
  'However, during some periods, Seth became one of Egypt’s great gods.',
  'His name appeared on Pharaohs like Seti I and II, who promoted him as a protector of Re, slaying Apopis with fierce power.'],
 ['Yet not everyone saw his strength positively.',
  'During the Second Intermediate Period, he was associated with evil forces like the Hyksos invaders, who linked him to their storm god Baal.',
  'This perception persisted through Greek and Roma

In [11]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in sentences:
    for s in p:
        audio = tts_model.generate_audio(voice_state, s)
        scipy.io.wavfile.write(f"Outputs\\output_{i}.wav", tts_model.sample_rate, audio.numpy())
        i += 1

In [12]:
wav_files = sorted([f for f in os.listdir("Outputs") if f.endswith('.wav')])
print(wav_files)

audio_data = []
seconds_per_paragraph = []
samplerate = None

for i in range(len(wav_files)):
    file_name = f"Outputs\\output_{i}.wav"
    
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("Outputs\\Final audio.wav", combined, samplerate)

['output_0.wav', 'output_1.wav', 'output_2.wav', 'output_3.wav', 'output_4.wav', 'output_5.wav', 'output_6.wav', 'output_7.wav', 'output_8.wav', 'output_9.wav']


In [13]:
durations = []
sentences_durations = []
images_needed = []
#seconds = []
i = 0
for p in sentences:
    duration_seconds = 0
    for s in p:
        file_path = f"Outputs\\output_{i}.wav"
        # Returns the sample rate (Fs) and the data as a NumPy array
        Fs, data = wavfile.read(file_path) 
        # The length of the data array divided by the sample rate gives the duration
        sentence_duration = len(data) / float(Fs)
        duration_seconds += sentence_duration
        sentences_durations.append(sentence_duration)
        i += 1
        os.remove(file_path)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    '''    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)'''


print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Durations of sentences: {sentences_durations}")
#print(f"Seconds for each image: {seconds}")

Durations of audio files: [23.040000000000003, 21.040000000000003, 15.04, 8.32]
Images needed for each paragraph: [3, 3, 2, 1]
Durations of sentences: [4.8, 10.8, 7.44, 8.96, 4.64, 7.44, 2.32, 8.08, 4.64, 8.32]


In [14]:
i = 0
sentence_start = 0
sentence_end = 0
image_text_chunks = []
seconds_for_chunk = []

# Use the already-split sentences from the first pass (list of lists)
for para_idx, para_sentences in enumerate(sentences_per_paragraph):
    # sentences_split is your original `sentences` list of lists
    para_sentence_list = sentences[para_idx]
    images_for_paragraph = images_needed[para_idx]
    images_for_paragraph = min(images_for_paragraph, len(para_sentence_list))

    total_sentences = len(para_sentence_list)
    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0
    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk_sentence_end = sentence_start + end  # absolute index into sentences_durations
        chunk_duration = sum(sentences_durations[sentence_start + start : sentence_start + end])
        seconds_for_chunk.append(chunk_duration)

        chunk = " ".join(para_sentence_list[start:end])
        groups.append(chunk)

        start = end

    sentence_start += total_sentences  # advance global pointer by paragraph's sentence count
    image_text_chunks.append(groups)

print(f"Text chunks for images: {image_text_chunks}")
print(f"Seconds for each chunk: {seconds_for_chunk}")

Text chunks for images: [['In ancient Egypt, the mysterious god Set ruled with power and chaos.', 'Worshipped since before recorded history, he was associated with storms, eclipses, thunderstorms, earthquakes, and the desert—making him both strong and unpredictable.', "His violent birth story, where he ripped himself from his mother's womb instead of being born normally, symbolized his chaotic nature."], ['Set held the was-sceptre and wore the Set animal as symbols, yet despite marriage to Nephthys, remained childless—a strange figure in the Egyptian pantheon.', 'However, during some periods, Seth became one of Egypt’s great gods.', 'His name appeared on Pharaohs like Seti I and II, who promoted him as a protector of Re, slaying Apopis with fierce power.'], ['Yet not everyone saw his strength positively. During the Second Intermediate Period, he was associated with evil forces like the Hyksos invaders, who linked him to their storm god Baal.', 'This perception persisted through Greek a

In [15]:
#name = Path(script_path).stem
fetched_images_ids = []
fetched_images_paths = []
for paragraph_chunks in image_text_chunks:
    for chunk in paragraph_chunks:
        text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
    
        with torch.no_grad():
            text_embedding = model.encode_text(text_tokens)
            text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

        embedding_list = text_embedding.cpu().numpy().tolist()[0]
        if is_landmark:
            query = "SELECT li.id, li.image_path FROM landmark_images as li JOIN landmarks as l ON li.landmark_id = l.id WHERE l.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"
        else:
            query = "SELECT pi.id, pi.image_path FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"


        
        with Session(engine) as session:
            result = session.execute(text(query), {"name": NAME, "embedding": embedding_list})
            images = result.fetchall()
            image_id = images[0][0]
            image_path = images[0][1]

            j = 0
            while image_id in fetched_images_ids:
                images.pop(0)
                if not images:
                    print("No more unique images available for this chunk.")
                    image_id = None
                    break
                image_id = images[0][0]
                image_path = images[0][1]
                j +=1
            fetched_images_ids.append(image_id)
            fetched_images_paths.append(image_path)

            
        print(f"Text chunk: {chunk}")
        print(f"Image ID: {image_id} on trial {j}")
        print(f"image path: {image_path}")

Text chunk: In ancient Egypt, the mysterious god Set ruled with power and chaos.
Image ID: 757 on trial 0
image path: data/video_generation/raw/pharaohs_images/Seth (God)/Seth god.jpg
Text chunk: Worshipped since before recorded history, he was associated with storms, eclipses, thunderstorms, earthquakes, and the desert—making him both strong and unpredictable.
Image ID: 760 on trial 1
image path: data/video_generation/raw/pharaohs_images/Seth (God)/Statue 2 angle 2.jpg
Text chunk: His violent birth story, where he ripped himself from his mother's womb instead of being born normally, symbolized his chaotic nature.
Image ID: 764 on trial 0
image path: data/video_generation/raw/pharaohs_images/Seth (God)/seth god of violence and chaos.jpg
Text chunk: Set held the was-sceptre and wore the Set animal as symbols, yet despite marriage to Nephthys, remained childless—a strange figure in the Egyptian pantheon.
Image ID: 753 on trial 1
image path: data/video_generation/raw/pharaohs_images/Seth 

In [16]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()
client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
if is_landmark:
    path = Path.joinpath(Path("data/video_generation/raw/landmark_images/"), NAME.replace(" ", "_").lower())
else:
    path = Path.joinpath(Path("data/video_generation/raw/pharaohs_images/"), NAME.replace(" ", "_").lower())
REMOTE_PREFIX = path

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

ordered_local_paths = []

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as executor:
    ordered_local_paths = list(
        executor.map(download_image, enumerate(fetched_images_paths))
    )

In [17]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))
image_files

[WindowsPath('temp_frames/0000.jpg'),
 WindowsPath('temp_frames/0001.jpg'),
 WindowsPath('temp_frames/0002.jpg'),
 WindowsPath('temp_frames/0003.jpg'),
 WindowsPath('temp_frames/0004.jpg'),
 WindowsPath('temp_frames/0005.jpg'),
 WindowsPath('temp_frames/0006.jpg'),
 WindowsPath('temp_frames/0007.jpg'),
 WindowsPath('temp_frames/0008.jpg')]

In [18]:
from PIL import Image as PILImage
for path in image_files:
    p = Path(path)
    try:
        with PILImage.open(p) as img:
            img = img.convert("RGB")  # handles AVIF, PNG, RGBA, etc.
            img.save(p, "JPEG", quality=95)
        print(f"Converted: {p}")
    except Exception as e:
        print(f"Failed to convert {p}: {e}")

Converted: temp_frames\0000.jpg
Converted: temp_frames\0001.jpg
Converted: temp_frames\0002.jpg
Converted: temp_frames\0003.jpg
Converted: temp_frames\0004.jpg
Converted: temp_frames\0005.jpg
Converted: temp_frames\0006.jpg
Converted: temp_frames\0007.jpg
Converted: temp_frames\0008.jpg


In [19]:
for img_path in Path("temp_frames").glob("*.jpg"):
    print(img_path, img_path.stat().st_size)
    try:
        with Image.open(img_path) as im:
            print("OK:", im.format, im.size)
    except Exception as e:
        print("Broken image:", e)

temp_frames\0000.jpg 41668
OK: JPEG (1100, 619)
temp_frames\0001.jpg 3412409
OK: JPEG (2988, 5312)
temp_frames\0002.jpg 282568
OK: JPEG (1000, 563)
temp_frames\0003.jpg 557564
OK: JPEG (939, 1252)
temp_frames\0004.jpg 158802
OK: JPEG (468, 626)
temp_frames\0005.jpg 171664
OK: JPEG (792, 969)
temp_frames\0006.jpg 207682
OK: JPEG (960, 960)
temp_frames\0007.jpg 215124
OK: JPEG (1000, 667)
temp_frames\0008.jpg 458531
OK: JPEG (1015, 679)


In [20]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 35
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds

# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks

'''
# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
            srt_blocks.append(srt_block)

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "Outputs/output_subtitles.srt")'''

'\n# ---------- MAIN FUNCTION ----------\ndef generate_srt(paragraphs, durations, output_path="subtitles.srt"):\n    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"\n\n    current_time = 0.0\n    subtitle_index = 1\n    srt_blocks = []\n\n    for paragraph, duration in zip(paragraphs, durations):\n        paragraph = normalize_text(paragraph)\n        sentences = split_into_sentences(paragraph)\n\n        # break into subtitle chunks\n        chunks = []\n        for sentence in sentences:\n            chunks.extend(split_long_text(sentence))\n\n        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)\n\n        for chunk in chunks:\n            chunk_char_count = len(chunk.replace("\n", ""))\n\n            # proportional timing\n            chunk_duration = max(\n                MIN_DURATION,\n                (chunk_char_count / total_chars) * duration\n            )\n\n            start_time = current_time\n            end_time = cur

In [21]:
def generate_srt(paragraphs, sentences_durations, output_path="subtitles.srt"):
    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []
    duration_index = 0  # pointer into flat sentences_durations list

    for paragraph in paragraphs:
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        for sentence in sentences:
            sentence_duration = sentences_durations[duration_index]
            duration_index += 1

            # Split long sentence into visual chunks
            chunks = split_long_text(sentence)
            total_chars = sum(len(c.replace("\n", "")) for c in chunks)

            for chunk in chunks:
                chunk_char_count = len(chunk.replace("\n", ""))

                # Proportional split only within the sentence now
                chunk_duration = max(
                    MIN_DURATION,
                    (chunk_char_count / total_chars) * sentence_duration
                )

                start_time = current_time
                end_time = current_time + chunk_duration

                srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
                srt_blocks.append(srt_block)
                current_time = end_time
                subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, sentences_durations, "Outputs/output_subtitles.srt")

SRT file saved to Outputs/output_subtitles.srt


In [22]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stderr)  # show FFmpeg error

    if result.returncode != 0:
        raise RuntimeError("FFmpeg failed")

def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080
):
    total_frames = int(duration * fps)

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)

    scaled_w = int(img_w * scale_factor)
    scaled_h = int(img_h * scale_factor)

    scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
    scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1

    max_x = scaled_w - target_w
    max_y = scaled_h - target_h

    threshold = 40  
    
    # Smooth easing expression
    #ease = f"(0.5-0.5*cos(PI*t/{duration}))"
    ease = f"(0.5-0.5*cos(PI*n/{total_frames}))"
    
    # Decide motion
    if (img_w / img_h) > (target_w / target_h):
        # Wider image → horizontal movement possible
        if max_x > threshold:
            x_expr = f"floor({max_x}*{ease})"
            y_expr = f"{max_y}/2"
            zoom_filter = ""
        else:
            # Not enough horizontal space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"
    else:
        # Taller image → vertical movement
        if max_y > threshold:
            x_expr = f"{max_x}/2"
            y_expr = f"floor({max_y}*{ease})"
            zoom_filter = ""
        else:
            # Not enough vertical space → subtle zoom
            zoom_filter = f",scale=iw*1.05:ih*1.05"
            x_expr = f"(iw-{target_w})/2"
            y_expr = f"(ih-{target_h})/2"

    vf = (
        f"scale={scaled_w}:{scaled_h}"
        f"{zoom_filter},"
        f"crop={target_w}:{target_h}:"
        f"x='{x_expr}':"
        f"y='{y_expr}'"
    )

    cmd = [
        "ffmpeg",
        "-y",
        "-loop", "1",
        "-framerate", str(fps),
        "-t", str(duration),
        "-i", image_path,
        "-vf", f"setsar=1,{vf}",
        "-frames:v", str(total_frames),
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",
        "-vsync", "cfr",
        "-video_track_timescale", "30",
        output_path
    ]

    run_ffmpeg(cmd)

In [23]:
def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)

    outputs = []

    for i, (img, dur) in enumerate(zip(image_files, durations)):
        out = f"{temp_dir}/clip_{i}.mp4"
        create_kenburns_clip(img, dur, out)
        outputs.append(out)

    return outputs

def concatenate_clips(clips, output_path):
    list_file = "Outputs/concat_list.txt"

    with open(list_file, "w") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c", "copy",       # ← stream copy, no re-encoding
        output_path
    ]

    run_ffmpeg(cmd)

In [24]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-fflags", "+genpts",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",
        "-fps_mode", "cfr",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('Outputs', f))

In [25]:
# 1. Generate Ken Burns clips
clips = generate_all_clips(image_files, seconds_for_chunk)

# 2. Concatenate
concatenated = "Outputs/combined.mp4"
concatenate_clips(clips, concatenated)

# 3. Add audio
with_audio = "Outputs/with_audio.mp4"
add_audio(concatenated, "Outputs/Final audio.wav", with_audio)

# 4. Add subtitles
final_output = "Outputs/final_video.mp4"
add_subtitles(with_audio, "Outputs/output_subtitles.srt", final_output)

#cleanup_files()

print("Done:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 4.8 -i temp_frames\0000.jpg -vf setsar=1,scale=1920:1080,scale=iw*1.05:ih*1.05,crop=1920:1080:x='(iw-1920)/2':y='(ih-1080)/2' -frames:v 144 -c:v h264_nvenc -preset p1 -pix_fmt yuv420p -vsync cfr -video_track_timescale 30 temp_clips/clip_0.mp4
ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enabl

In [26]:
cleanup_files()